# Tests: `fasterai.core.schedule` (source `nbs/core/schedules.ipynb`)

In [ ]:
from fastcore.test import *
from fastai.callback.schedule import sched_lin
from fasterai.core.schedule import *

In [ ]:
from fastcore.test import *

# one_shot (start_pct=0.5): 0 before start, jumps to 1 at start
test_eq(one_shot.progress(0.0), 0.0)
test_eq(one_shot.progress(0.49), 0.0)
test_close(one_shot.progress(1.0), 1.0, eps=1e-6)
one_shot.reset()

# agp (start_pct=0.2): cubic at pct=0.5
# normalized pos = (0.5 - 0.2) / (1.0 - 0.2) = 0.375
# agp formula: 0 + (0 - 1) * (1 - 0.375)^3 = 1 - 0.625^3 ≈ 0.756
test_close(agp.progress(0.5), 0.756, eps=0.01)
agp.reset()

# cos: at midpoint ≈ 0.5
test_close(cos.progress(0.5), 0.5, eps=0.05)
cos.reset()

# lin: linear → 0.5 at midpoint
test_close(lin.progress(0.5), 0.5, eps=1e-6)
lin.reset()

# changed property tracks state transitions
sched = Schedule(sched_lin)
_ = sched.progress(0.0)
test_eq(sched.changed, False)  # 0→0, no change
_ = sched.progress(0.5)
test_eq(sched.changed, True)  # 0→0.5, changed
sched.after_step()
test_eq(sched.changed, False)  # previous updated
sched.reset()

# reset restores initial state
sched2 = Schedule(sched_lin)
_ = sched2.progress(0.8)
sched2.reset()
test_eq(sched2._current_progress, 0.0)
test_eq(sched2._previous_progress, 0.0)

# Custom start_pct
late = Schedule(sched_lin, start_pct=0.5)
test_eq(late.progress(0.0), 0.0)   # before start
test_eq(late.progress(0.49), 0.0)  # still before start
test_close(late.progress(0.75), 0.5, eps=0.01)
late.reset()

# available_schedules returns list
scheds = available_schedules()
assert 'one_shot' in scheds
assert 'agp' in scheds
assert 'cos' in scheds
assert len(scheds) > 0

# dsd returns to 0 at end
test_close(dsd.progress(1.0), 0.0, eps=1e-6)
dsd.reset()